In [0]:
stdf = spark.read.csv(
    "/Workspace/Users/falisty1612@gmail.com/SteamMarketAnalysis/steam.csv"
    , header=True
    ,inferSchema=True
)
stdf.createOrReplaceTempView("stdf_raw")

In [0]:
import pyspark.sql.functions as F

stdf.select([F.count(F.col(a)).alias(a) for a in stdf.columns]).show()

In [0]:
#castowanie price na typ double
stdf = stdf.withColumn("price", F.col("price").cast("double"))

#zamiana foramtu daty
stdf = stdf.withColumn("release_date", F.to_date("release_date", "yyyy-MM-dd" ))



In [0]:
#rozbijanie kolumn ze stringów na listy

stdf = stdf.withColumn("platforms", F.split(F.col("platforms"), ";"))

stdf = stdf.withColumn("categories", F.split(F.col("categories"), ";"))

stdf = stdf.withColumn("genres", F.split(F.col("genres"), ";"))

stdf = stdf.withColumn("steamspy_tags", F.split(F.col("steamspy_tags"), ";"))




In [0]:
#dodanie kolumn min i max dla owners

stdf = stdf.withColumn("split_owners", F.split(F.col("owners"), "-"))

stdf = stdf.withColumn("min_owners", F.col("split_owners")[0].cast("int"))
stdf = stdf.withColumn("max_owners", F.col("split_owners")[1].cast("int"))

stdf = stdf.withColumn("avg_owners", (F.col("max_owners")+F.col("min_owners"))/2)

In [0]:
display(stdf)

In [0]:
#średnia cena za godzinę gry

stdf = stdf.withColumn("price_per_hour", F.when(F.col("price") == 0, 0 ).otherwise(F.col("median_playtime") / F.col("price")))

In [0]:
#engagement = median_playtime / avg_owners

stdf = stdf.withColumn("engagement", F.col("median_playtime")/F.col("avg_owners"))

In [0]:
stdf = stdf.withColumn("review_ratio", F.col("positive_ratings")-F.col("negative_ratings"))

In [0]:
stdf_genres = (
    stdf.select(
        "appid"
        ,"name"
        ,"price"
        ,"median_playtime"
        ,"avg_owners"
        ,F.explode("genres").alias("genre")
    )
)

display(stdf_genres)